In [1]:
import joblib
import cv2
import cmapy
from PIL import Image
from msi_visual.extraction import get_extraction_mz_list
import tqdm
import numpy as np
from collections import defaultdict
from msi_visual.supervised.annotations import get_img, get_visualization#, get_detection_mask

from msi_visual.normalization import total_ion_count
from msi_atlas.annotations import get_dataset
import numpy as np
from argparse import Namespace
from pathlib import Path
import joblib
array = np.array
import os

path = r"E:\MSImaging-data\_msi_visual\Extractions\atlas_verification"
extraction_args = eval(
    open(
        Path(path) /
        "args.txt").read())
extraction_mzs = extraction_args.mzs

paths = [(path + "\\0.npy", 1), (path + "\\2.npy", 0), (path + "\\1.npy", 2), (path + "\\3.npy", 3)]
X, y, slide_labels, label_encoder = get_dataset(r"NRL4485-s2_reannotation_23-12-24_PAHJ.json", 
                                  paths, subsample=20, normalization=total_ion_count)

y = np.array(y)
data = {"extraction_args": extraction_args, "X": X, "y": y, "label_encoder": label_encoder}


FileNotFoundError: [Errno 2] No such file or directory: 'NRL4485-s2_reannotation_23-12-24_PAHJ.json'

In [ ]:
# mzs = [1160.8, 1161.8, 1162.0, 1163.0, 1165.0]
# mzs += [1160.8, 1162.0, 1163.0, 1165.0, 1141.0, 1142.0, 1143.0]
#interesting mzs: [808.2, 809.2]

mzs = [1160.8, 1161.8, 1162.0, 1163.0]

mzs = [1179.73081311,1207.75938587,480.30869475,1180.73943434,1207.78445573,540.0534138,865.50307848,866.50730465,857.51706004,858.51944986,858.52058908,859.52412677]

mzs = list(set((mzs)))
mzs.sort()
print(mzs)
intensities = defaultdict(list)
ions = {}
gallery_images = []
for mouse_index, (path, index) in tqdm.tqdm(enumerate(paths)):
    img = get_img(path)
    pred = xgb_model.predict(img.reshape(-1, img.shape[-1])).reshape(img.shape[:2])        
    mask = img.max(axis=-1)
    for mz in mzs:
        ion_image = img[:, :, extraction_mzs.index(mz)]
        #ion_image[pred == 0] = 0
        ion_image[mask == 0] = 0
        ion_image = ion_image.transpose(1, 0)[::-1, :]
        ion_image[masks[mouse_index] == 0] = 0

        if mouse_index == 1:
            ion_image = ion_image[:, ::-1]

        
        intensities[mz].append(np.percentile(ion_image[:], 99))
        ions[(mz, path)] = ion_image
    del img, pred
    



gather = True
all_images = []
for mz in mzs:
    gallery_images = []
    for mouse_index, (path, index) in tqdm.tqdm(enumerate(paths)):
        ion_image = ions[(mz, path)]
        ion_image = ion_image / np.max(intensities[mz])
        ion_image[ion_image > 1] = 1
        ion_image = (ion_image * 255).astype(np.uint8)
        ion_image = cv2.applyColorMap(ion_image, cmapy.cmap('magma'))[:, :, ::-1].copy()


        ion = cv2.resize(ion_image, (262, 286))
        font = cv2.FONT_HERSHEY_SIMPLEX
        bottomLeftCornerOfText = (5, 20)
        fontScale = 0.5
        fontColor = (255, 255, 255)
        thickness = 1
        lineType = 1
        ion = cv2.putText(ion, f"{mz:.3f}",
                                bottomLeftCornerOfText,
                                font,
                                fontScale,
                                fontColor,
                                thickness,
                                lineType)
        if "0.npy" in path or "1.npy" in path:
            ion = cv2.putText(ion, f"A7KO", (5, 50), font, fontScale, fontColor, thickness, lineType)
        gallery_images.append(ion)

    gallery_images = np.hstack(gallery_images)
    all_images.append(gallery_images)
    
all_images = np.vstack(all_images)
display(Image.fromarray(all_images))
            

